# Crypto Order Book Sampler
**Author:** Ralph Cuevas   
**Environment:** Google Colab  

---

## Project Overview

This notebook samples live cryptocurrency order book data from the **Coinbase Advanced Trade public API** (no API key required).  
It was originally written against the Binance API, but Binance returns HTTP 451 (geo-blocked) from US-based servers such as Google Colab.
Coinbase exposes an equivalent public order book endpoint with no restrictions.
It calculates key market microstructure metrics from the bid/ask order book and logs them automatically on a schedule, saving results to a dated CSV file.

### Metrics Captured
| Metric | Description |
|--------|-------------|
| `mp` | Mid price (best ask price) |
| `tav` | Total ask volume within price threshold |
| `tbv` | Total bid volume within price threshold |
| `ac` | Ask capital (price × quantity) within threshold |
| `bc` | Bid capital (price × quantity) within threshold |
| `timestamp` | Unix timestamp of sample |

### How It Works
1. Connects to Coinbase's public order book endpoint
2. Calculates a price range threshold (0.36%) around the best bid/ask
3. Aggregates volume and capital within that range
4. Logs each sample to a CSV file every N seconds automatically

---

## Step 1 — Install Dependencies

In [ ]:
# All libraries used are available in Colab by default — no installs needed
import requests
import pandas as pd
import numpy as np
import csv
import time
import sched
from datetime import date
from IPython.display import display, clear_output

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Step 2 — Configuration

Set your trading pair and sampling parameters here.  
Coinbase uses hyphenated pairs (e.g. `BTC-USD`, `ETH-USD`, `SOL-USD`).

In [ ]:
# --- Configuration ---
SYMBOL       = 'BTC-USD'   # Coinbase product ID (e.g. 'BTC-USD', 'ETH-USD', 'SOL-USD')
RANGE        = 0.0036      # Price threshold: 0.36% around best bid/ask
INTERVAL_SEC = 5           # How often to sample (in seconds)
MAX_SAMPLES  = 20          # Total number of samples to collect (set to None to run indefinitely)
ORDER_BOOK_LEVEL = 2       # Coinbase level 2 = full aggregated order book

# Coinbase public order book endpoint — no API key required
COINBASE_URL = f'https://api.exchange.coinbase.com/products/{SYMBOL}/book?level={ORDER_BOOK_LEVEL}'

print(f"Configuration set.")
print(f"  Symbol         : {SYMBOL}")
print(f"  Price range    : {RANGE*100:.2f}%")
print(f"  Sample interval: {INTERVAL_SEC}s")
print(f"  Max samples    : {MAX_SAMPLES}")
print(f"  Endpoint       : {COINBASE_URL}")

Configuration set.
  Symbol         : BTC-USD
  Price range    : 0.36%
  Sample interval: 5s
  Max samples    : 20
  Endpoint       : https://api.exchange.coinbase.com/products/BTC-USD/book?level=2


## Step 3 — Order Book Helper Functions

These functions process the raw order book data and calculate the key metrics.

> **Note on Coinbase response format:** Coinbase returns each order book entry as `[price, size, num_orders]`.  
> We only use `price` and `size` (equivalent to Binance's `price` and `qty`), so we slice `[:2]` on each entry.

In [ ]:
def fetch_order_book(url):
    """
    Fetches the current order book from Coinbase.
    Coinbase entries are [price, size, num_orders] — we take price and size only.
    Returns separate DataFrames for asks and bids.
    """
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    df_asks = pd.DataFrame(
        [entry[:2] for entry in data['asks']],
        columns=['price', 'qty']
    ).astype(float)

    df_bids = pd.DataFrame(
        [entry[:2] for entry in data['bids']],
        columns=['price', 'qty']
    ).astype(float)

    return df_asks, df_bids


def sum_qty_asks(df):
    """
    Total ask quantity (volume) within the price threshold above best ask.
    """
    top_price = df['price'].min()  # best ask is the lowest ask
    threshold = top_price * RANGE
    return df[df['price'] <= top_price + threshold]['qty'].sum()


def sum_qty_bids(df):
    """
    Total bid quantity (volume) within the price threshold below best bid.
    """
    top_price = df['price'].max()  # best bid is the highest bid
    threshold = top_price * RANGE
    return df[df['price'] >= top_price - threshold]['qty'].sum()


def sum_asks_cap(df):
    """
    Total ask capital (price x qty) within threshold — represents sell-side liquidity in USD.
    """
    top_price = df['price'].min()
    threshold = top_price * RANGE
    filtered = df[df['price'] <= top_price + threshold].copy()
    filtered['capital'] = filtered['price'] * filtered['qty']
    return filtered['capital'].sum()


def sum_bids_cap(df):
    """
    Total bid capital (price x qty) within threshold — represents buy-side liquidity in USD.
    """
    top_price = df['price'].max()
    threshold = top_price * RANGE
    filtered = df[df['price'] >= top_price - threshold].copy()
    filtered['capital'] = filtered['price'] * filtered['qty']
    return filtered['capital'].sum()


def get_snapshot(url):
    """
    Fetches the order book and returns a single row of computed metrics.
    """
    df_asks, df_bids = fetch_order_book(url)

    mp  = round(df_asks['price'].min(), 4)   # mid price (best ask)
    tav = round(sum_qty_asks(df_asks), 4)    # total ask volume
    tbv = round(sum_qty_bids(df_bids), 4)    # total bid volume
    ac  = round(sum_asks_cap(df_asks), 4)    # ask capital
    bc  = round(sum_bids_cap(df_bids), 4)    # bid capital
    ts  = int(time.time())                   # unix timestamp

    return {'mp': mp, 'tav': tav, 'tbv': tbv, 'ac': ac, 'bc': bc, 'timestamp': ts}


print("Helper functions defined.")

Helper functions defined.


## Step 4 — Test a Single Snapshot

Run one sample to confirm the API connection is working before starting the scheduler.

In [ ]:
snapshot = get_snapshot(COINBASE_URL)

print(f"Live snapshot for {SYMBOL}:")
print(f"  Mid Price (mp)  : ${snapshot['mp']:,.2f}")
print(f"  Ask Volume (tav): {snapshot['tav']} BTC")
print(f"  Bid Volume (tbv): {snapshot['tbv']} BTC")
print(f"  Ask Capital (ac): ${snapshot['ac']:,.2f}")
print(f"  Bid Capital (bc): ${snapshot['bc']:,.2f}")
print(f"  Timestamp       : {snapshot['timestamp']}")

Live snapshot for BTC-USD:
  Mid Price (mp)  : $61,751.76
  Ask Volume (tav): 66.611 BTC
  Bid Volume (tbv): 67.2233 BTC
  Ask Capital (ac): $4,119,542.86
  Bid Capital (bc): $4,144,540.07
  Timestamp       : 1780811900


## Step 5 — Automated Scheduler

This cell runs the sampler automatically every `INTERVAL_SEC` seconds and saves all results to a dated CSV file.  
It will stop automatically after `MAX_SAMPLES` samples are collected.

In [ ]:
# Output filename based on today's date
today = date.today()
filename = today.strftime("%Y-%m-%d") + ".csv"
sample_count = [0]  # mutable counter for use inside nested function
results = []        # store rows for live display

def sample_and_log(scheduler):
    """
    Collects one order book snapshot, writes it to CSV, and re-schedules itself.
    """
    if MAX_SAMPLES is not None and sample_count[0] >= MAX_SAMPLES:
        print(f"\nCompleted {MAX_SAMPLES} samples. Data saved to '{filename}'.")
        return

    # Schedule next call before doing work (keeps interval consistent)
    scheduler.enter(INTERVAL_SEC, 1, sample_and_log, (scheduler,))

    try:
        row = get_snapshot(COINBASE_URL)
        results.append(row)
        sample_count[0] += 1

        # Write to CSV
        write_header = sample_count[0] == 1  # only write header on first row
        with open(filename, mode='a', newline='') as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=['mp', 'tav', 'tbv', 'ac', 'bc', 'timestamp'])
            if write_header:
                writer.writeheader()
            writer.writerow(row)

        # Live display in Colab
        clear_output(wait=True)
        print(f"Sampling {SYMBOL} — {sample_count[0]}/{MAX_SAMPLES} samples collected")
        print(f"Output file: {filename}\n")
        df_live = pd.DataFrame(results)
        df_live['timestamp'] = pd.to_datetime(df_live['timestamp'], unit='s')
        display(df_live.tail(10))

    except Exception as e:
        print(f"Error on sample {sample_count[0]}: {e}")


# Initialize and run the scheduler
print(f"Starting sampler for {SYMBOL}...")
print(f"Collecting {MAX_SAMPLES} samples every {INTERVAL_SEC} seconds.\n")

my_scheduler = sched.scheduler(time.time, time.sleep)
my_scheduler.enter(0, 1, sample_and_log, (my_scheduler,))
my_scheduler.run()

Sampling BTC-USD — 20/20 samples collected
Output file: 2026-06-07.csv



,mp,tav,tbv,ac,bc,timestamp
10,61723.99,62.9085,68.8327,3.888653e+06,4.242047e+06,2026-06-07 05:59:10
11,61730.37,66.6063,65.4545,4.117770e+06,4.034454e+06,2026-06-07 05:59:15
12,61736.12,61.1348,70.1411,3.780042e+06,4.323854e+06,2026-06-07 05:59:20
13,61729.09,63.8201,72.5570,3.945725e+06,4.472231e+06,2026-06-07 05:59:26
14,61729.09,65.2236,72.7810,4.032410e+06,4.486035e+06,2026-06-07 05:59:31
15,61729.09,64.7322,73.4282,4.002021e+06,4.525951e+06,2026-06-07 05:59:36
16,61729.00,64.0858,71.7001,3.962066e+06,4.419441e+06,2026-06-07 05:59:41
17,61724.01,60.5478,69.2619,3.742835e+06,4.268604e+06,2026-06-07 05:59:45
18,61726.98,63.3994,70.2883,3.919643e+06,4.332258e+06,2026-06-07 05:59:50
19,61732.67,60.6492,66.7933,3.749477e+06,4.117469e+06,2026-06-07 05:59:55



Completed 20 samples. Data saved to '2026-06-07.csv'.


## Step 6 — Review & Export Collected Data

In [ ]:
# Load the saved CSV and display a summary
df = pd.read_csv(filename)
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

print(f"Data collected: {len(df)} rows")
print(f"Time range: {df['timestamp'].min()} → {df['timestamp'].max()}\n")
print("Summary statistics:")
display(df.describe())
display(df)

Data collected: 20 rows
Time range: 2026-06-07 05:58:20 → 2026-06-07 05:59:55

Summary statistics:


,mp,tav,tbv,ac,bc,timestamp
count,20.000000,20.000000,20.00000,2.000000e+01,2.000000e+01,20
mean,61737.802500,64.686775,69.21218,3.999739e+06,4.266534e+06,2026-06-07 05:59:07.700000256
min,61723.990000,60.547800,63.34890,3.742835e+06,3.904886e+06,2026-06-07 05:58:20
25%,61729.090000,63.276675,67.81745,3.911896e+06,4.181325e+06,2026-06-07 05:58:43.750000128
50%,61733.930000,65.360500,69.18265,4.041768e+06,4.264625e+06,2026-06-07 05:59:07.500000
75%,61750.455000,66.276000,71.01470,4.098558e+06,4.378292e+06,2026-06-07 05:59:32.249999872
max,61754.900000,67.992700,73.42820,4.204907e+06,4.525951e+06,2026-06-07 05:59:55
std,10.789096,2.222190,2.54555,1.378734e+05,1.567801e+05,NaN


,mp,tav,tbv,ac,bc,timestamp
0,61751.76,66.6110,67.2233,4.119543e+06,4.144540e+06,2026-06-07 05:58:20
1,61751.76,65.6007,70.9865,4.057183e+06,4.376763e+06,2026-06-07 05:58:25
2,61751.77,67.2414,69.3062,4.158560e+06,4.273074e+06,2026-06-07 05:58:30
3,61752.06,67.4460,66.8222,4.171218e+06,4.119876e+06,2026-06-07 05:58:35
4,61754.90,65.4974,69.1034,4.051126e+06,4.260645e+06,2026-06-07 05:58:40
5,61750.02,66.1659,68.0155,4.092155e+06,4.193587e+06,2026-06-07 05:58:45
6,61745.52,67.9927,68.5015,4.204907e+06,4.223398e+06,2026-06-07 05:58:50
7,61739.99,62.7951,63.3489,3.882845e+06,3.904886e+06,2026-06-07 05:58:55
8,61733.93,65.5258,71.0993,4.051309e+06,4.382880e+06,2026-06-07 05:59:00
9,61733.93,65.7518,68.5987,4.065294e+06,4.228678e+06,2026-06-07 05:59:05


In [ ]:
# Download the CSV to your local machine (Colab only)
from google.colab import files
files.download(filename)
print(f"'{filename}' downloaded.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'2026-06-07.csv' downloaded.
